# Module 06, Part 1: Tokenization and self-supervised pretraining

This notebook is the runnable companion to Part 1 of the Module 06 learning content.
It checks, with code:

1. Byte-pair encoding, built from scratch, reproduces the merge table in the text, and
   words it never saw are built from pieces it did.
2. A tokenizer that was trained on one kind of text cuts other kinds of text into more
   pieces, and more pieces cost more.
3. The next-token loss is a plain cross-entropy over a table of probabilities, and its
   starting value is $\log V$ for a vocabulary of size $V$.
4. The contrastive (InfoNCE) loss is a classification problem over the batch. Pretraining
   with it on **unlabelled** digits gives features that a classifier can use with very
   few labels.

Everything runs on a CPU in under a minute. The experiments are small on purpose, so
every number can be reproduced; they show mechanics, not how a model with billions of
parameters behaves.

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter

INK, BODY, MUTED = "#14161B", "#4E5563", "#7C838C"
NAVY, GREEN, AMBER, KEY, RULE = "#1C3D6E", "#0E6B57", "#8A6A18", "#7A2E8A", "#D9DCE2"

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": RULE, "axes.labelcolor": BODY, "xtick.color": BODY, "ytick.color": BODY,
    "axes.grid": True, "grid.color": RULE, "font.size": 10,
})

torch.manual_seed(0)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.15.0+cpu


## 1. Byte-pair encoding from scratch

Start from characters plus an end-of-word marker `</w>`. Repeat: count every adjacent
pair of symbols (weighted by how often the word occurs), merge the most frequent pair
into one new symbol, and record the merge. Ties are broken by taking the largest
`(count, pair)` so the result never depends on dictionary order.

In [2]:
from collections import Counter

END = "</w>"


def word_to_symbols(word):
    return tuple(word) + (END,)


def pair_counts(vocab):
    """Count adjacent symbol pairs, weighted by how often each word occurs."""
    counts = Counter()
    for symbols, freq in vocab.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += freq
    return counts


def apply_merge(vocab, pair):
    merged = pair[0] + pair[1]
    new_vocab = {}
    for symbols, freq in vocab.items():
        out, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                out.append(merged)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        new_vocab[tuple(out)] = new_vocab.get(tuple(out), 0) + freq
    return new_vocab


def train_bpe(word_counts, n_merges):
    """Return the list of merges as (pair, count_at_the_time).

    Ties are broken by taking the largest (count, pair) so the result is
    deterministic and does not depend on dictionary order.
    """
    vocab = {word_to_symbols(w): c for w, c in word_counts.items()}
    merges = []
    for _ in range(n_merges):
        counts = pair_counts(vocab)
        if not counts:
            break
        pair = max(counts, key=lambda p: (counts[p], p))
        merges.append((pair, counts[pair]))
        vocab = apply_merge(vocab, pair)
    return merges


def encode_word(word, merges):
    """Apply the learned merges, in the order they were learned, to a new word."""
    symbols = word_to_symbols(word)
    for pair, _ in merges:
        merged = pair[0] + pair[1]
        out, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                out.append(merged)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        symbols = tuple(out)
    return list(symbols)


def word_counts_from_text(text):
    return dict(Counter(text.lower().split()))

In [3]:
corpus = {"low": 5, "lower": 2, "lowest": 2, "newer": 6, "newest": 6,
          "wider": 3, "widest": 3, "wide": 4}
merges = train_bpe(corpus, n_merges=10)

print(f"{'merge':>5s}  {'pair merged':<14s}{'new symbol':<10s}{'count':>6s}")
for i, ((a, b), n) in enumerate(merges, 1):
    print(f"{i:5d}  {a + ' + ' + b:<14s}{a + b:<10s}{n:6d}")

expected = ["we", "ne", "newe", "t</w>", "st</w>", "r</w>", "wi", "wid", "wide", "lo"]
assert [a + b for (a, b), _ in merges] == expected
print("\nThe ten merges match the table in the learning content.")

merge  pair merged   new symbol count
    1  w + e         we            16
    2  n + e         ne            12
    3  ne + we       newe          12
    4  t + </w>      t</w>         11
    5  s + t</w>     st</w>        11
    6  r + </w>      r</w>         11
    7  w + i         wi            10
    8  wi + d        wid           10
    9  wid + e       wide          10
   10  l + o         lo             9

The ten merges match the table in the learning content.


### Words the corpus never contained

Apply the ten merges, in the order they were learned, to new words.

In [4]:
for w in ["slowest", "lowering", "newer", "widening"]:
    print(f"{w:10s} -> {encode_word(w, merges)}")

slowest    -> ['s', 'lo', 'we', 'st</w>']
lowering   -> ['lo', 'we', 'r', 'i', 'n', 'g', '</w>']
newer      -> ['newe', 'r</w>']
widening   -> ['wide', 'n', 'i', 'n', 'g', '</w>']


"slowest" is built from pieces that were learned (`lo`, `we`, `st</w>`) even though the
word never appeared. "lowering" is covered at the front by learned pieces, but the
corpus never contained "ing", so the ending falls back to single characters. The same
is true of "widening". Production tokenizers work on bytes, so nothing is truly
unknown, but the effect is the same: text that looks like the training text is cut
into few, long pieces, and text that does not is cut into many short ones.

## 2. Token counts depend on what the tokenizer was trained on

Train 60 merges on a short paragraph of ordinary English prose, then count how many
tokens per word three different texts need.

In [5]:
train_text = """
the model reads a sentence one token at a time and the tokenizer decides how the sentence
is cut into pieces a short common word is usually one piece while a long rare word is
split into several pieces the cost of a model grows with the number of pieces so the way
a word is cut matters when the text is about weather or cooking or travel the pieces are
long and the sequence is short when the text is about something the tokenizer rarely saw
the pieces are short and the sequence is long every extra piece is one more position
that attention has to compare with every other position and that is why the length of
the sequence is the thing we pay for
"""
bpe = train_bpe(word_counts_from_text(train_text), n_merges=60)

samples = {
    "same style (English prose)": "the weather is long and the model is cheap to read in the morning",
    "code": "def forward self x return self linear x relu",
    "rare technical words": "electroencephalography photolithography spectrophotometry",
}
print(f"{'text':28s}{'words':>6s}{'tokens':>8s}{'tokens per word':>17s}")
for name, text in samples.items():
    words = text.split()
    n_tok = sum(len(encode_word(w, bpe)) for w in words)
    print(f"{name:28s}{len(words):6d}{n_tok:8d}{n_tok / len(words):17.2f}")

text                         words  tokens  tokens per word
same style (English prose)      14      33             2.36
code                             9      38             4.22
rare technical words             3      48            16.00


Words that look like the training text cost about two to three tokens each, and the
rare technical words cost several times as many. Token counts, not word counts, set the
cost, because each token is one position in attention. The tokenizer is also part of
the model: the model learned what each token ID means, so a model must always be loaded
with the tokenizer it was trained with.

## 3. The next-token loss

Predicting the next token is a classification over the vocabulary at every position.
The loss for a document $x_1, \ldots, x_T$ is
$-\sum_t \log p(x_t \mid x_1, \ldots, x_{t-1})$, averaged over positions in practice.

The simplest possible predictor is a bigram table: $p(x_t \mid x_{t-1})$ estimated from
counts. We compute its loss on a held-out sentence and compare it with the loss of
knowing nothing, which is $\log V$.

In [6]:
chars = sorted(set(train_text))
V = len(chars)
idx = {c: i for i, c in enumerate(chars)}
ids = torch.tensor([idx[c] for c in train_text])

counts = torch.ones(V, V)                       # add-one smoothing so no probability is zero
for a, b in zip(ids[:-1], ids[1:]):
    counts[a, b] += 1
P = counts / counts.sum(1, keepdim=True)        # P[a, b] = p(next = b | current = a)

held_out = "the sentence is cut into pieces and the model reads the pieces"
h = torch.tensor([idx[c] for c in held_out])
bigram_loss = -torch.log(P[h[:-1], h[1:]]).mean().item()
print(f"vocabulary size V = {V}")
print(f"loss of knowing nothing (log V): {math.log(V):.3f}")
print(f"loss of the bigram table:        {bigram_loss:.3f}")

logits = torch.log(P[h[:-1]])                   # the same numbers through F.cross_entropy
print(f"same value through F.cross_entropy: {F.cross_entropy(logits, h[1:]).item():.3f}")

vocabulary size V = 27
loss of knowing nothing (log V): 3.296
loss of the bigram table:        2.090
same value through F.cross_entropy: 2.090


Even a table of one-character contexts is clearly better than knowing nothing, and the
loss is exactly what `F.cross_entropy` computes. A neural language model is a much
better predictor of $p(x_t \mid \text{everything before})$, trained by the same loss.
Nothing in this loss was labelled by a person: the label for each position is the next
character of the text itself.

## 4. The contrastive loss, and pretraining without labels

For a batch of $N$ items, let $s_{ij}$ be the similarity between the embedding of view
$i$ and the embedding of the other view of item $j$. The loss for item $i$ is

$$\mathcal{L}_i = -\log \frac{\exp(s_{ii}/\tau)}{\sum_{j=1}^{N} \exp(s_{ij}/\tau)}$$

That is a softmax cross-entropy in which the right answer for item $i$ is index $i$.
So it can be written with `F.cross_entropy` on a similarity matrix. We check that
against the formula written out directly.

In [7]:
def info_nce(z1, z2, tau=0.2):
    """InfoNCE for two views of the same items. z1[i] and z2[i] are two views of item i."""
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    sim = z1 @ z2.T / tau                        # sim[i, j] = s_ij / tau
    return F.cross_entropy(sim, torch.arange(len(z1)))

z1, z2 = torch.randn(8, 16), torch.randn(8, 16)
direct = 0.0
a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
for i in range(8):
    s = (a[i] @ b.T) / 0.2
    direct += -(s[i] - torch.logsumexp(s, 0)).item()
print(f"formula written out: {direct / 8:.6f}")
print(f"info_nce():          {info_nce(z1, z2).item():.6f}")
print(f"loss with random embeddings is about log N = {math.log(8):.3f}")

formula written out: 2.279353
info_nce():          2.279352
loss with random embeddings is about log N = 2.079


### Pretrain on unlabelled digits, then use very few labels

Take the 1,257 training digits **without their labels**. For each image make two views
by adding noise, scaling the brightness and sometimes shifting it by one pixel. Train an
encoder so the two views of one image have similar embeddings and views of different
images do not. Then freeze the encoder and train only a linear classifier on 50
**labelled** images, and compare with the same classifier on a random, untrained
encoder and on raw pixels.

In [8]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

d = load_digits()
X, y = d.data / 16.0, d.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xtr_t, Xte_t = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Xte, dtype=torch.float32)

def view(x, g):
    """One random view: brightness scale, noise, and a shift of one pixel half of the time."""
    scale = 0.7 + 0.6 * torch.rand(len(x), 1, generator=g)
    out = x * scale + 0.15 * torch.randn(x.shape, generator=g)
    img = out.view(-1, 8, 8)
    shift = torch.randint(-1, 2, (len(x),), generator=g)
    img = torch.stack([torch.roll(im, int(s), dims=1) for im, s in zip(img, shift)])
    return img.reshape(-1, 64)

def make_encoder():
    return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64))

def pretrain_contrastive(seed, steps=400, batch=128):
    torch.manual_seed(seed)
    enc = make_encoder()
    opt = torch.optim.Adam(enc.parameters(), 1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        i = torch.randint(0, len(Xtr_t), (batch,), generator=g)
        loss = info_nce(enc(view(Xtr_t[i], g)), enc(view(Xtr_t[i], g)))
        opt.zero_grad(); loss.backward(); opt.step()
    return enc, loss.item()

def probe_accuracy(features_fn, n_labels=50, seed=0):
    """Fit a logistic regression on n_labels labelled training images; test on the held-out images."""
    rng = np.random.RandomState(seed)
    pick = rng.choice(len(Xtr), n_labels, replace=False)
    with torch.no_grad():
        ftr, fte = features_fn(Xtr_t[pick]).numpy(), features_fn(Xte_t).numpy()
    scaler = StandardScaler().fit(ftr)           # put every feature on the same scale before the linear fit
    clf = LogisticRegression(max_iter=3000).fit(scaler.transform(ftr), ytr[pick])
    return clf.score(scaler.transform(fte), yte)

rows = {"raw pixels": [], "random encoder": [], "contrastive pretrained": []}
for seed in range(5):
    torch.manual_seed(100 + seed)
    random_enc = make_encoder()
    enc, final_loss = pretrain_contrastive(seed)
    rows["raw pixels"].append(probe_accuracy(lambda x: x, seed=seed))
    rows["random encoder"].append(probe_accuracy(random_enc, seed=seed))
    rows["contrastive pretrained"].append(probe_accuracy(enc, seed=seed))

print("Linear classifier trained on 50 labelled digits, mean accuracy over 5 seeds")
for name, v in rows.items():
    print(f"  {name:24s} {np.mean(v):.3f}   (min {np.min(v):.3f}, max {np.max(v):.3f})")

Linear classifier trained on 50 labelled digits, mean accuracy over 5 seeds
  raw pixels               0.774   (min 0.722, max 0.815)
  random encoder           0.766   (min 0.709, max 0.817)
  contrastive pretrained   0.842   (min 0.794, max 0.889)


Read this result with some care.

- **Pretraining helped, but modestly.** The pretrained encoder reaches 0.841 against 0.774
  for raw pixels and 0.766 for a random encoder, so about seven points from labels it
  never saw. A random encoder is no better than pixels: the gain comes from what
  training on the unlabelled images taught the encoder, not from the extra layers.
- **Digits are an easy case for raw pixels**, because an 8 by 8 digit is already close to
  a good representation. The gain from pretraining grows when the raw input is far from
  a useful representation, as it is for text.
- **These settings were chosen after looking at held-out accuracy.** A first attempt
  without scaling the features before the linear fit gave 0.720 for the pretrained
  encoder, below pixels. Linear probes are sensitive to feature scale, so scale them.
  Treat the seven points as indicative and not as a benchmark.

## Try it

1. Add 40 more merges to the eight-word corpus experiment. When does the encoding of
   "lowering" stop falling back to single characters, and what does the vocabulary cost?
2. Train the bigram table on your own paragraph and compute its loss on a sentence in a
   different style. How far does the loss rise?
3. Change the temperature `tau` in `info_nce` to 0.05 and to 1.0 and rerun the
   pretraining. Which gives the better probe accuracy, and why might a very small
   temperature be harder to train?
4. Use a batch of 16 instead of 128 in the contrastive pretraining. A larger batch gives
   more wrong answers to push away. Does the smaller batch give a worse probe?

## What this notebook showed

- Byte-pair encoding is a short loop of count, merge and record, and the merges it
  learns decide how every later text is cut.
- A tokenizer is part of the model. Text unlike the training text is cut into more
  pieces, and every piece costs a position in attention.
- Next-token prediction is cross-entropy against a label that is part of the text. The
  contrastive loss is cross-entropy against the other view of the same item. Neither
  needs a human label.
- Pretraining on unlabelled data gives features that need few labels to use. That is the
  reuse the rest of the module builds on.